

```
# Ce texte est au format code
```

# Nouvelle section

In [ ]:
import os #to access to some files
import json#to load my kagle file
from zipfile import ZipFile#extract filein zip
from PIL import Image #load data# Charger l'image
#image = Image.open('chemin/vers/votre/image.png')
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg#Lecture de l'image: Utilisez la fonction mpimg.imread pour lire une image ? partir d'un fichier.
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers,models



In [ ]:
!pip install kaggle

In [ ]:
#setup kaggle key as environment variable
kaggle_credentails=json.load(open('kaggle.json'))
os.environ["KAGGLE_USERNAME"]=kaggle_credentails["username"]
os.environ["KAGGLE_KEY"]=kaggle_credentails["key"]


In [ ]:
!kaggle datasets download -d abdallahalidev/plantvillage-dataset

In [ ]:
!ls


In [ ]:
#unzip the data
with ZipFile('plantvillage-dataset.zip','r') as z:
  z.extractall()

In [ ]:
print(os.listdir('plantvillage dataset'))
print(len(os.listdir('plantvillage dataset/color')))
print(len(os.listdir('plantvillage dataset/grayscale')))
print(len(os.listdir('plantvillage dataset/segmented')))


In [ ]:
base_dir="plantvillage dataset/color"

In [ ]:
Image_path="/content/plantvillage dataset/color/Apple___Cedar_apple_rust/025b2b9a-0ec4-4132-96ac-7f2832d0db4a___FREC_C.Rust 3655.JPG"
#Read the image
img=mpimg.imread(Image_path)


#Display the image
plt.imshow(img)
plt.axis('off')
plt.show()


In [ ]:
#image Parameters
img_size=244
batch_size=32

In [ ]:
#the dataset
data_gen=ImageDataGenerator(rescale=1./255,validation_split=0.2)#""reseving some image for testing""
"""1. ImageDataGenerator
C'est une classe de Keras qui permet de pr?traiter les images avant de les fournir au mod?le. Elle est utile pour normaliser les images et appliquer des transformations comme la rotation, le zoom, ou le flip.

2. rescale=1./255
Les valeurs des pixels dans les images sont g?n?ralement comprises entre 0 et 255 (8 bits par canal).
En divisant chaque pixel par 255, on ram?ne ces valeurs dans l?intervalle [0,1], ce qui facilite l?entra?nement du mod?le en am?liorant la convergence.

3. validation_split=0.2
Cela signifie que 20 % des images seront r?serv?es pour la validation et ne seront pas utilis?es dans l'entra?nement.

80 % seront utilis?es pour l'entra?nement.
20 % seront utilis?es pour la validation."""

In [ ]:
#train dataset
data_gen_train = data_gen.flow_from_directory(
    base_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    subset='training',
    class_mode='categorical'
)
"""Charge les images depuis un r?pertoire (base_dir) contenant des sous-dossiers, o? chaque sous-dossier repr?sente une classe d?images.
Cette m?thode est utile lorsque les images sont organis?es sous forme de r?pertoires par classes
batch_size=batch_size

Nombre d?images charg?es par lot pendant l?entra?nement.
Par exemple, si batch_size=32, alors 32 images seront charg?es ? chaque it?ration.
Comme validation_split=0.2 a ?t? d?fini dans ImageDataGenerator,
cette ligne signifie qu'on charge uniquement les 80 % des images r?serv?es pour l'entra?nement.
class_mode='categorical'

Indique que les classes sont mutuellement exclusives (ex : classification d?images en plusieurs cat?gories comme "chien", "chat", "oiseau").
Cela signifie que la sortie du mod?le sera sous forme one-hot encoded"""


In [ ]:
#test dataset
data_gen_validation=data_gen.flow_from_directory(
    base_dir,
    target_size=(img_size,img_size),
    batch_size=batch_size,
    subset='validation',
    class_mode='categorical')

In [ ]:
model=models.Sequential()
model.add(layers.Conv2D(32,(3,3),activation='relu',input_shape=(img_size,img_size,3)))
model.add(layers.MaxPooling2D((2,2)))
model.add(layers.Conv2D(64,(3,3),activation='relu'))
model.add(layers.MaxPooling2D((2,2)))
model.add(layers.Flatten())
model.add(layers.Dropout(0.5))
model.add(layers.Dense(64,activation='relu'))
model.add(layers.Dense(38,activation='softmax'))
"""
Premi?re convolution (32 filtres) ? Apprend des motifs simples (bords, textures).
Deuxi?me convolution (64 filtres) ? Apprend des motifs plus complexes (formes, contours).
Plus on avance, plus les filtres augmentent ? Le mod?le apprend plus de d?tails.
model.add(layers.Dense(38,activation='softmax'))
38 neurones car on a 38 classes ? pr?dire.
Activation Softmax car on fait une classification multi-classes (probabilit?s de chaque classe qui doivent donner 1).
########################################################################################################################
Couche	Forme des donn?es
Image d?entr?e	(150, 150, 3) (hauteur, largeur, canaux RVB)
Apr?s Conv2D(32, (3,3))	(150, 150, 32) (chaque filtre produit une carte 2D)
Apr?s MaxPooling2D(2,2)	(75, 75, 32) (la taille est r?duite de moiti?)
Apr?s Conv2D(64, (3,3))	(75, 75, 64) (on ajoute 64 nouvelles cartes)
Apr?s MaxPooling2D(2,2)	(37, 37, 64)
Apr?s Flatten()	Vecteur 1D : (37?37?64 = 87 296)
###################################################################################################################"
***ReLU est pr?f?r?e car elle ?vite le probl?me du gradient qui dispara?t, contrairement ? sigmoid et tanh.
----Dans les premi?res couches, les valeurs du gradient sont proches de 0 ? Le r?seau n?apprend plus bien !
Pourquoi ? Parce que sigmoid ?crase les valeurs extr?mes vers 0 ou 1, rendant ses d?riv?es proches de 0.
----avec ReLU, les gradients restent bien plus grands, m?me dans les premi?res couches.
Pourquoi ? Parce que ReLU garde des d?riv?es ? 1 pour les valeurs positives ? Les poids continuent ? s?adapter correctement.

***Si on mettait ReLU en derni?re couche : Probl?me :

La sortie ne serait pas normalis?e.
Certaines valeurs pourraient ?tre n?gatives ou sup?rieures ? 1.
Impossible d?interpr?ter les valeurs comme probabilit?s.
----Avec Softmax, chaque sortie est comprise entre 0 et 1, et la somme fait 1.
"""

In [ ]:
model.summary()


In [ ]:
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

"""
****Optimiseur	Avantages	Inconv?nients
SGD	Simple, fonctionne bien pour les petits datasets	Lent pour des r?seaux profonds
RMSprop	S?adapte au gradient, efficace pour CNNs	Peut converger trop rapidement
Adam (choisi ici) ?	Meilleure convergence, ajustement automatique du learning rate	Peut parfois ?tre trop rapide et sauter des minima
?? Pourquoi Adam ici ?
Parce qu?il fonctionne bien sur les CNNs et r?duit le besoin de tuner manuellement le learning rate.
****loss='categorical_crossentropy' ?
La fonction de perte (loss) mesure l??cart entre les pr?dictions du mod?le et la v?rit? terrain (y_true).
Elle guide l'optimiseur pour ajuster les poids du r?seau.
****Les m?triques ?valuent les performances du mod?le.

accuracy
?

Pour classification multi-classes, elle compte le pourcentage de bonnes pr?dictions.
"""


In [ ]:
history=model.fit(data_gen_train,
                  steps_per_epoch=data_gen_train.samples,
                  epochs=5,
                  validation_data=data_gen_validation,
                  validation_steps=data_gen_validation.samples)

In [ ]:
val_loss,val_acc=model.evaluate(data_gen_validation)

In [ ]:
val_loss,val_acc

In [ ]:
plt.plot(history.history['accuracy'],label='accuracy')
plt.plot(history.history['val_accuracy'],label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5,1])
plt.legend()
plt.show()

In [ ]:
#create a mapping from class indices to class names
class_indices={v: k for k,v in data_gen_train.class_indices.items()}

In [ ]:
#function to load and preprocessing the image using PIllow
def load_and_preprocess_image(image_path,target_size=(244,244)):
  #load the image
  img=Image.open(image_path)
  #reseize the image
  img=img.resize(target_size)
  #convert the image to a numpy array
  img_array=np.array(img)
  #expand the dimensions of the array
  img_array=np.expand_dims(img_array,axis=0)
  #normalize the image
  img_array=img_array.astype('float32')/255
  return img_array

"""Un mod?le CNN attend une entr?e sous la forme :


(batch_size,height,width,channels)
Dimension	Signification	Exemple (batch_size=32, image=224x224x3)
1?? axe (batch_size)	Nombre d'images dans le batch	32 (batch de 32 images)
2? axe (height)	Hauteur de l?image	224 pixels
3? axe (width)	Largeur de l?image	224 pixels
4? axe (channels)	Nombre de canaux (couleurs)	3 (Rouge, Vert, Bleu - RGB)"""
#function to predict
def predict_image(image_path,model):
  #load and preprocess the image
  preprocessing_img=load_and_preprocess_image(image_path)
  prediction=model.predict(preprocessing_img)
  predicted_class_index=np.argmax(prediction,axis=1)[0]
  """np.argmax(prediction, axis=1) retourne l'indice de la classe ayant la probabilit?
  la plus ?lev?e pour chaque ?chantillon dans prediction. Cela suppose que prediction est
  une matrice 2D o? chaque ligne correspond ? un ?chantillon et chaque colonne ? une classe."""
  predicted_class_name=class_indices[predicted_class_index]
  return predicted_class_name
""""On utilise PIL pour manipuler des images de mani?re simple et efficace, notamment pour des t?ches comme le chargement, la conversion de formats et le redimensionnement. cv2 est plus puissant pour le traitement avanc? des images (filtrage, d?tection d'objets, segmentation, etc.).
"J?utilise PIL plut?t que cv2 car PIL est plus simple pour charger, modifier et afficher des images.
C'est aussi plus compatible avec TensorFlow et Keras car il charge les images en RGB (contrairement ? cv2 qui utilise BGR).
Si j?avais besoin de traitement d?image avanc? comme la d?tection d?objets ou la segmentation, j?utiliserais cv2.
"""""


In [ ]:
print(data_gen_train)

In [ ]:
class_indices

In [ ]:
#saving the class names as json file
json.dump(class_indices,open('class_indices.json','w'))

In [ ]:
#example of image
image_path='tomato.jpg'
predicted_class_name=predict_image(image_path,model)
print(f'The predicted class is {predicted_class_name}')

In [ ]:
# Stocker les maladies dans une liste
all_diseases = list(class_indices.values())

# Afficher la liste
print("Liste des maladies :", all_diseases)


In [ ]:
%pip install transformers torch


In [ ]:
model

In [ ]:
!pip install langchain-groq


In [ ]:
from langchain_groq import ChatGroq
import numpy as np
from PIL import Image

def initialize_llm():
    """Initialise Groq LLM avec LangChain."""
    return ChatGroq(
        temperature=0.7,
        groq_api_key="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",
        model_name="llama3-8b-8192"
    )

# ? Pr?traitement de l'image pour la classification CNN
def load_and_preprocess_image(image_path, target_size=(244, 244)):
    img = Image.open(image_path).convert("RGB")  # Convertir en RGB
    img = img.resize(target_size)
    img_array = np.array(img) / 255.0  # Normalisation entre 0 et 1
    img_array = np.expand_dims(img_array, axis=0)  # Ajouter la dimension batch
    return img_array

# ? Effectuer la pr?diction CNN et g?n?rer le traitement avec Groq
def predict_and_generate_treatment(image_path, model, class_indices, llm):
    preprocessed_img = load_and_preprocess_image(image_path)
    print(f"? Image charg?e avec succ?s ! Forme : {preprocessed_img.shape}")

    # ? Pr?diction avec le mod?le CNN (d?j? charg?)
    prediction = model.predict(preprocessed_img)
    predicted_class_index = np.argmax(prediction, axis=1)[0]
    predicted_class_name = class_indices[predicted_class_index]

    # ? G?n?rer une r?ponse avec Groq
    prompt = f"Quels sont les traitements pour la maladie {predicted_class_name} sur cette plante ?"
    treatment = llm.invoke(prompt)

    return predicted_class_name, treatment

# ? Dictionnaire des classes du mod?le CNN
class_indices = {
    0: 'Apple___Apple_scab', 1: 'Apple___Black_rot', 2: 'Apple___Cedar_apple_rust', 3: 'Apple___healthy',
    4: 'Blueberry___healthy', 5: 'Cherry_(including_sour)___Powdery_mildew', 6: 'Cherry_(including_sour)___healthy',
    7: 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 8: 'Corn_(maize)___Common_rust_',
    9: 'Corn_(maize)___Northern_Leaf_Blight', 10: 'Corn_(maize)___healthy',
    11: 'Grape___Black_rot', 12: 'Grape___Esca_(Black_Measles)', 13: 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    14: 'Grape___healthy', 15: 'Orange___Haunglongbing_(Citrus_greening)', 16: 'Peach___Bacterial_spot',
    17: 'Peach___healthy', 18: 'Pepper,_bell___Bacterial_spot', 19: 'Pepper,_bell___healthy',
    20: 'Potato___Early_blight', 21: 'Potato___Late_blight', 22: 'Potato___healthy',
    23: 'Raspberry___healthy', 24: 'Soybean___healthy', 25: 'Squash___Powdery_mildew',
    26: 'Strawberry___Leaf_scorch', 27: 'Strawberry___healthy', 28: 'Tomato___Bacterial_spot',
    29: 'Tomato___Early_blight', 30: 'Tomato___Late_blight', 31: 'Tomato___Leaf_Mold',
    32: 'Tomato___Septoria_leaf_spot', 33: 'Tomato___Spider_mites Two-spotted_spider_mite',
    34: 'Tomato___Target_Spot', 35: 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    36: 'Tomato___Tomato_mosaic_virus', 37: 'Tomato___healthy'
}

# ? Charger et pr?dire la maladie avec le mod?le CNN existant
image_path = "apple.jpg"
llm = initialize_llm()  # Initialiser le mod?le Groq
predicted_disease, treatment_recommendation = predict_and_generate_treatment(image_path, model, class_indices, llm)

# ? Affichage du r?sultat
print(f"?? **Plante d?tect?e :** {predicted_disease.split('___')[0]}")
print(f"?? **Maladie d?tect?e :** {predicted_disease.split('___')[1]}")
print(f"?? **Traitement recommand? :** {treatment_recommendation}")


In [ ]:
model.save('my_model1.h5')

In [ ]:
!pip install streamlit tensorflow pillow

